In [1]:
import requests
import json
import time
from datetime import datetime

def fetch_mcat_posts_pullpush(limit=500):
    url = "https://api.pullpush.io/reddit/search/submission/"
    
    # We will collect data here
    all_posts = []
    
    # Start looking from "now" backwards
    current_time = int(time.time())
    
    print(f"Starting scrape from PullPush for r/MCAT...")

    while len(all_posts) < limit:
        params = {
            "subreddit": "MCAT",
            "size": 100,              # Max allowed per request
            "before": current_time,   # Get posts OLDER than this timestamp
            "sort": "desc",
            "sort_type": "created_utc"
        }

        try:
            response = requests.get(url, params=params)
            data = response.json()['data']
            
            if not data:
                break # No more posts found

            for post in data:
                # Filter for useful text posts only
                if 'selftext' in post and post['selftext'] != "" and post['selftext'] != "[removed]":
                    
                    # Construct the Reddit URL manually since archives might not have it
                    permalink = post.get('permalink', f"/r/MCAT/comments/{post['id']}/")
                    full_url = f"https://www.reddit.com{permalink}"

                    post_obj = {
                        "id": post['id'],
                        "title": post['title'],
                        "content": post['selftext'],
                        "url": full_url,
                        "created_utc": post['created_utc'],
                        # PullPush returns posts, not comments. 
                        # For a portfolio, just the post text is usually enough 
                        # to demonstrate RAG.
                        "combined_text": f"Title: {post['title']}\nContent: {post['selftext']}" 
                    }
                    all_posts.append(post_obj)

            # Update timestamp to the last post's time to get the next batch
            current_time = data[-1]['created_utc']
            
            print(f"Collected {len(all_posts)} posts so far...")
            time.sleep(1) # Be polite to the API

        except Exception as e:
            print(f"Error: {e}")
            break

    # Save to file
    with open("mcat_pullpush_data.json", "w", encoding='utf-8') as f:
        json.dump(all_posts, f, indent=4)
    
    print(f"Finished! Saved {len(all_posts)} posts.")

# Run it
fetch_mcat_posts_pullpush(limit=500)

Starting scrape from PullPush for r/MCAT...
Collected 70 posts so far...
Collected 142 posts so far...
Collected 217 posts so far...
Collected 305 posts so far...
Collected 390 posts so far...
Collected 481 posts so far...
Collected 559 posts so far...
Finished! Saved 559 posts.


In [5]:
import requests
import json
import time
import random

# 1. Load your existing posts
try:
    with open("mcat_top_1000.json", "r", encoding='utf-8') as f:
        posts = json.load(f)
except FileNotFoundError:
    print("Error: Could not find mcat_top_1000.json. Run the previous script first!")
    exit()

print(f"Loaded {len(posts)} posts. Starting comment enrichment...")

enriched_data = []

# Headers are CRITICAL. If you don't use a browser-like User-Agent, Reddit blocks you immediately.
headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

for i, post in enumerate(posts):
    # Construct the JSON URL
    # post['url'] usually looks like https://www.reddit.com/r/MCAT/comments/xyz/title/
    # We need https://www.reddit.com/r/MCAT/comments/xyz/title.json
    
    json_url = post['url']
    if json_url.endswith("/"):
        json_url = json_url[:-1] + ".json"
    else:
        json_url = json_url + ".json"

    try:
        response = requests.get(json_url, headers=headers, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            
            # Reddit JSON is a list of 2 items: 
            # [0] is the Post info, [1] is the Comments listing
            comments_data = data[1]['data']['children']
            
            extracted_comments = []
            
            # Grab top 5 comments (Reddit sorts by 'Best' by default, which is perfect)
            for comment in comments_data[:5]:
                c_data = comment.get('data', {})
                
                # Filter out empty, deleted, or bot comments
                if 'body' in c_data and c_data['body'] not in ["[deleted]", "[removed]"]:
                    extracted_comments.append({
                        "author": c_data.get('author', 'anon'),
                        "score": c_data.get('score', 0),
                        "body": c_data['body']
                    })
            
            # Add comments to our post object
            post['top_comments'] = extracted_comments
            enriched_data.append(post)
            
            print(f"[{i+1}/{len(posts)}] Scraped {len(extracted_comments)} comments for: {post['title'][:30]}...")
            
        elif response.status_code == 429:
            print("Rate Limit Hit! Sleeping for 10 seconds...")
            time.sleep(10)
        else:
            print(f"Failed ({response.status_code}) for {post['url']}")

    except Exception as e:
        print(f"Error processing {post['id']}: {e}")

    # SLEEP IS MANDATORY - Vary it slightly to look human
    time.sleep(random.uniform(2.0, 10.0))

    # Save progress every 50 posts (in case script crashes)
    if (i + 1) % 50 == 0:
        with open("mcat_enriched_final.json", "w", encoding='utf-8') as f:
            json.dump(enriched_data, f, indent=4)
        print(">>> Progress saved to mcat_enriched_final.json")

# Final Save
with open("mcat_enriched_final.json", "w", encoding='utf-8') as f:
    json.dump(enriched_data, f, indent=4)

print("Done! You now have posts AND their best answers.")

Loaded 559 posts. Starting comment enrichment...
[1/559] Scraped 2 comments for: 5/23 Void?...
[2/559] Scraped 5 comments for: How did you memorize solubilit...
[3/559] Scraped 5 comments for: Anybody else take the scored f...
[4/559] Scraped 2 comments for: AAMC question bank or Uworld q...
[5/559] Scraped 2 comments for: uworld FLs??...
[6/559] Scraped 0 comments for: Deadline for MCAT to apply for...
[7/559] Scraped 2 comments for: MCAT Study Tools/Tips?...
[8/559] Scraped 2 comments for: Link to Mr. Pankow P/S deck?...
[9/559] Scraped 1 comments for: Square roots!...
[10/559] Scraped 4 comments for: Is the inferior colliculus res...
[11/559] Scraped 5 comments for: Slow down!...
[12/559] Scraped 1 comments for: Should I wait another cycle ?...
[13/559] Scraped 2 comments for: Uworld account....
[14/559] Scraped 1 comments for: Mcat study tips/ schedule...
[15/559] Scraped 4 comments for: CARS resources...
[16/559] Scraped 2 comments for: Vent...
[17/559] Scraped 1 comments for: tmd